# STOP/APO screening in *C. elegans* — AB, MS, C, D and E

**Unified, reproducible Google Colab workflow**

This notebook runs the two independent comparisons for each lineage:

1. **Individual APO recognition:** APO positions are compared with all other biological positions in the same normalized lineage window. This scan is skipped when a lineage has no APO positions.
2. **STOP vs DIVIDE:** STOP positions are compared with DIVIDE positions. APO is kept separate and is not merged into STOP for this comparison.

`NULL` positions are retained in the positional coordinate system where required by the lineage-specific normalization, but are never treated as biological states or included in identity comparisons. The identity threshold is **strictly below 50%** for every required cross-class comparison.

## Required input files

Upload these six files:

- `TR_CATALOG_RAW.tsv` — the authoritative TR source. It is included at the root of `ARCHIVIO_TR_LINEAGE_COMPLETO_CON_SPECCHIETTO.zip` and in the original TRF raw archive. Use this file only; do not substitute derived/final catalogs.
- `AB_BOTTOM_UP_TOP_DOWN.txt`
- `MS_BOTTOM_UP_TOP_DOWN.txt`
- `C_BOTTOM_UP_TOP_DOWN.txt`
- `D_BOTTOM_UP_TOP_DOWN.txt`
- `E_BOTTOM_UP_TOP_DOWN.txt`

The five lineage text files are included in the validated lineage archive. Extract them before uploading. The upload resolver accepts Colab duplicate suffixes such as `(1)`.

## TR screening rules

- The input catalog is `TR_CATALOG_RAW.tsv` only.
- Period-1 repeats are excluded.
- For periods 2 and 3, every complete monomer must contain at least one adenine.
- Only complete monomers are retained; incomplete terminal nucleotides are excluded.
- A maximum of **250 complete monomers** is applied to every lineage scan.
- Each lineage/root additionally requires a TR window long enough for its own normalized positions.

The notebook does not modify the source files. It writes new CSV results and one ZIP archive.

## Expected lineage checks

| Lineage | STOP | DIVIDE | APO |
|---|---:|---:|---:|
| AB | 391 | 486 (including founder AB) | 96 |
| MS | 80 | 93 | 14 |
| C | 47 | 47 | 1 |
| D | 20 | 19 | 0 |
| E | 20 | 19 | 0 |

For AB, the normalized 32-root analysis excludes the founder and contains 942 biological positions: 455 DIVIDE, 391 STOP and 96 APO.


## 1. Upload and identify the six required input files

In [ ]:
from google.colab import files
from pathlib import Path
import re

BASE = Path('/content')
RESULTS_DIR = BASE / 'STOP_APO_AB_MS_C_D_E_results'
RESULTS_DIR.mkdir(exist_ok=True)

uploaded = files.upload()

REQUIRED_INPUTS = [
    'TR_CATALOG_RAW.tsv',
    'AB_BOTTOM_UP_TOP_DOWN.txt',
    'MS_BOTTOM_UP_TOP_DOWN.txt',
    'C_BOTTOM_UP_TOP_DOWN.txt',
    'D_BOTTOM_UP_TOP_DOWN.txt',
    'E_BOTTOM_UP_TOP_DOWN.txt',
]

def normalized_upload_name(filename):
    # Remove only Colab's duplicate suffix, e.g. "file (1).txt".
    return re.sub(r' \(\d+\)(?=\.[^.]+$)', '', filename)

available = {}
for filename in BASE.iterdir():
    if filename.is_file():
        available.setdefault(normalized_upload_name(filename.name), []).append(filename)

INPUT_FILES = {}
missing = []
for expected in REQUIRED_INPUTS:
    matches = available.get(expected, [])
    if len(matches) != 1:
        if not matches:
            missing.append(expected)
        else:
            raise RuntimeError(
                f'Multiple files resolve to {expected}: {[p.name for p in matches]}'
            )
    else:
        INPUT_FILES[expected] = matches[0]

if missing:
    raise FileNotFoundError('Missing required input file(s): ' + ', '.join(missing))

print('Validated input files:')
for name, path in INPUT_FILES.items():
    print(f'- {name}: {path.stat().st_size:,} bytes')

TR_FILE = INPUT_FILES['TR_CATALOG_RAW.tsv']
AB_FILE = INPUT_FILES['AB_BOTTOM_UP_TOP_DOWN.txt']


## 2. AB lineage — reconstruction and structural validation

AB is analyzed over the 32 established five-generation roots. The founder `AB` is included in the complete-tree state validation, but not in the 32-root normalized windows.

In [ ]:
import pandas as pd
import numpy as np

AB_FILE = INPUT_FILES['AB_BOTTOM_UP_TOP_DOWN.txt']
lines = AB_FILE.read_text(
    encoding="utf-8-sig", errors="replace"
).splitlines()

def extract_section(start_title, end_title=None):
    start = next(
        i for i, line in enumerate(lines)
        if line.strip() == start_title
    )
    if end_title is None:
        end = len(lines)
    else:
        end = next(
            i for i in range(start + 1, len(lines))
            if lines[i].strip() == end_title
        )
    return {
        line.strip()[3:]
        for line in lines[start + 1:end]
        if line.strip() == "AB" or line.strip().startswith("AB ")
    }

albero_completo = extract_section(
    "-- ALBERO COMPLETO --",
    "========== STOP (391) =========="
)
STOP_AB = extract_section(
    "========== STOP (391) ==========",
    "========== DIVIDE (486) =========="
)
DIVIDE_AB = extract_section(
    "========== DIVIDE (486) ==========",
    "========== APO INFERITE (92) =========="
)
APO_INFERITE = extract_section(
    "========== APO INFERITE (92) ==========",
    "========== APO-HERM. NOMINATE (4) =========="
)
APO_NOMINATE = extract_section(
    "========== APO-HERM. NOMINATE (4) =========="
)

states_AB = {}
for node in albero_completo:
    if node in DIVIDE_AB:
        states_AB[node] = "DIVIDE"
    elif node in STOP_AB:
        states_AB[node] = "STOP"
    elif node in APO_INFERITE or node in APO_NOMINATE:
        states_AB[node] = "APO"
    else:
        states_AB[node] = "UNCLASSIFIED"

radici_AB = sorted(
    node for node in albero_completo if len(node) == 5
)

summary = []
for root in radici_AB:
    nodes = [n for n in albero_completo if n.startswith(root)]
    summary.append({
        "Root": root,
        "Nodes": len(nodes),
        "DIVIDE": sum(states_AB[n] == "DIVIDE" for n in nodes),
        "STOP": sum(states_AB[n] == "STOP" for n in nodes),
        "APO": sum(states_AB[n] == "APO" for n in nodes),
        "Unclassified": sum(states_AB[n] == "UNCLASSIFIED" for n in nodes)
    })

df_AB_32_verification = pd.DataFrame(summary)

assert len(radici_AB) == 32, f"Expected 32 roots; found {len(radici_AB)}"
assert len(STOP_AB) == 391, f"Expected 391 STOP nodes; found {len(STOP_AB)}"
assert len(DIVIDE_AB) == 486, f"Expected 486 DIVIDE nodes; found {len(DIVIDE_AB)}"
assert len(APO_INFERITE) == 92, f"Expected 92 inferred APO nodes; found {len(APO_INFERITE)}"
assert len(APO_NOMINATE) == 4, f"Expected 4 nominated APO nodes; found {len(APO_NOMINATE)}"
assert not any(v == "UNCLASSIFIED" for v in states_AB.values()), "Unclassified AB nodes detected"

print("Complete-tree nodes:", len(albero_completo))
print("Roots:", len(radici_AB))
print("STOP:", len(STOP_AB))
print("DIVIDE:", len(DIVIDE_AB))
print("Inferred APO:", len(APO_INFERITE))
print("Nominated APO:", len(APO_NOMINATE))
print("Unclassified nodes:", sum(v == "UNCLASSIFIED" for v in states_AB.values()))
display(df_AB_32_verification)

In [ ]:
# The founder AB precedes the 32 root subtrees.
biological_nodes = set(albero_completo) | {"AB"}
complete_states = dict(states_AB)
complete_states["AB"] = "DIVIDE"

divide_without_children = []
divide_with_one_child = []
terminal_nodes_with_children = []

for node, state in complete_states.items():
    if node == "AB":
        children = [child for child in ("a", "p") if child in biological_nodes]
    else:
        children = [node + suffix for suffix in "aplrdv"
                    if node + suffix in biological_nodes]

    if state == "DIVIDE":
        if len(children) == 0:
            divide_without_children.append(node)
        elif len(children) != 2:
            divide_with_one_child.append((node, children))
    elif state in {"STOP", "APO"} and children:
        terminal_nodes_with_children.append((node, children))

assert not divide_without_children, (
    f"DIVIDE nodes without children: {divide_without_children[:10]}"
)
assert not divide_with_one_child, (
    f"DIVIDE nodes with only one child: {divide_with_one_child[:10]}"
)
assert not terminal_nodes_with_children, (
    f"Terminal nodes with biological children: {terminal_nodes_with_children[:10]}"
)

print("Structural checks passed.")
print("DIVIDE nodes without children:", len(divide_without_children))
print("DIVIDE nodes with other than two children:", len(divide_with_one_child))
print("STOP/APO nodes with children:", len(terminal_nodes_with_children))

In [ ]:
normalized_rows = []

for root in radici_AB:
    root_nodes = sorted(
        (node for node in albero_completo if node.startswith(root)),
        key=lambda node: (len(node), node)
    )

    positions = []
    for node in root_nodes:
        state = states_AB[node]
        positions.append({
            "Root": root,
            "Node": node,
            "State": state,
            "Position_type": "BIOLOGICAL"
        })

        # Preserve the two virtual NULL positions associated with APO.
        if state == "APO":
            positions.extend([
                {
                    "Root": root,
                    "Node": node + "_NULL1",
                    "State": "NULL",
                    "Position_type": "VIRTUAL"
                },
                {
                    "Root": root,
                    "Node": node + "_NULL2",
                    "State": "NULL",
                    "Position_type": "VIRTUAL"
                }
            ])

    for position, record in enumerate(positions, start=1):
        record["Position"] = position
        normalized_rows.append(record)

df_AB_normalized = pd.DataFrame(normalized_rows)

# Remove only trailing virtual NULL positions; retain internal NULL positions.
final_windows = []
for root in radici_AB:
    df_root = (
        df_AB_normalized[df_AB_normalized["Root"] == root]
        .sort_values("Position")
        .copy()
    )
    biological_positions = df_root.loc[
        df_root["Position_type"] == "BIOLOGICAL", "Position"
    ]
    last_biological_position = int(biological_positions.max())

    df_root = df_root[
        df_root["Position"] <= last_biological_position
    ].copy()
    df_root["Final_position"] = range(1, len(df_root) + 1)
    final_windows.append(df_root)

df_AB_normalized_final = pd.concat(
    final_windows, ignore_index=True
)

root_summary = (
    df_AB_normalized_final
    .groupby("Root")
    .agg(
        Window_positions=("Final_position", "size"),
        Biological_positions=("Position_type",
                              lambda x: (x == "BIOLOGICAL").sum()),
        Internal_NULL=("Position_type",
                      lambda x: (x == "VIRTUAL").sum())
    )
    .reset_index()
)

biological_count = int(
    (df_AB_normalized_final["Position_type"] == "BIOLOGICAL").sum()
)
apo_count = int(
    (df_AB_normalized_final["State"] == "APO").sum()
)
divide_count = int(
    (df_AB_normalized_final["State"] == "DIVIDE").sum()
)
stop_count = int(
    (df_AB_normalized_final["State"] == "STOP").sum()
)

assert biological_count == 942, biological_count
assert divide_count == 455, divide_count
assert stop_count == 391, stop_count
assert apo_count == 96, apo_count

print("Normalized roots:", len(root_summary))
print("Biological positions:", biological_count)
print("DIVIDE:", divide_count)
print("STOP:", stop_count)
print("APO:", apo_count)
print("Virtual NULL positions retained:", int(
    (df_AB_normalized_final["Position_type"] == "VIRTUAL").sum()
))
display(root_summary)

## 3. AB lineage — prepare the TR catalog and apply the 250-monomer limit

The AB workflow retains its lineage-specific minimum-window requirement. The common TR rules are applied before scanning: Period greater than 1, complete monomers only, the established adenine condition for periods 2 and 3, and no more than 250 complete monomers.

In [ ]:
TR_FILE = INPUT_FILES['TR_CATALOG_RAW.tsv']
TR_RAW = pd.read_csv(
    TR_FILE,
    sep="\t",
    encoding="utf-8-sig",
    low_memory=False
)

required_columns = {
    "Chromosome", "Start", "End", "Period", "Repeat_Sequence"
}
missing_columns = required_columns - set(TR_RAW.columns)
if missing_columns:
    raise ValueError(
        "Missing columns in TR_CATALOG_RAW.tsv: "
        + ", ".join(sorted(missing_columns))
    )

TR_RAW["Period"] = pd.to_numeric(TR_RAW["Period"], errors="coerce")
TR_RAW = TR_RAW[
    TR_RAW["Period"].notna() & (TR_RAW["Period"] > 0)
].copy()
TR_RAW["Period"] = TR_RAW["Period"].astype(int)
TR_RAW["Repeat_Sequence"] = (
    TR_RAW["Repeat_Sequence"].fillna("").astype(str).str.upper()
)

TR_RAW["Original_length"] = TR_RAW["Repeat_Sequence"].str.len()
TR_RAW["Incomplete_tail"] = (
    TR_RAW["Original_length"] % TR_RAW["Period"]
)
TR_RAW["Complete_sequence"] = [
    sequence[:len(sequence) - tail] if tail else sequence
    for sequence, tail in zip(
        TR_RAW["Repeat_Sequence"],
        TR_RAW["Incomplete_tail"]
    )
]
TR_RAW["N_monomers"] = (
    TR_RAW["Complete_sequence"].str.len() // TR_RAW["Period"]
).astype(int)

minimum_window_length = int(
    root_summary["Window_positions"].min()
)

TR_FILTERED = TR_RAW[
    (TR_RAW["Period"] > 1) &
    (TR_RAW["N_monomers"] >= minimum_window_length)
].copy()

# Established period-2/period-3 adenine criterion.
short_period = TR_FILTERED[
    TR_FILTERED["Period"].isin([2, 3])
].copy()

short_period["A_in_every_monomer"] = [
    all(
        "A" in sequence[i:i + period]
        for i in range(0, len(sequence), period)
    )
    for sequence, period in zip(
        short_period["Complete_sequence"],
        short_period["Period"]
    )
]

TR_FILTERED = pd.concat(
    [
        TR_FILTERED[TR_FILTERED["Period"] >= 4],
        short_period[short_period["A_in_every_monomer"]]
    ],
    ignore_index=True
)

print("Raw catalog records:", len(TR_RAW))
print("Minimum normalized window length:", minimum_window_length)
print("TR candidates after length and period filters:", len(TR_FILTERED))
print("Complete monomers — min/max:",
      int(TR_FILTERED["N_monomers"].min()),
      int(TR_FILTERED["N_monomers"].max()))
print("TR with incomplete terminal sequence removed:",
      int((TR_FILTERED["Incomplete_tail"] > 0).sum()))

In [ ]:
MAX_MONOMERS = 250

TR_FILTERED_250 = (
    TR_FILTERED[
        TR_FILTERED["N_monomers"] <= MAX_MONOMERS
    ]
    .copy()
    .reset_index(drop=True)
)

print("TR before maximum-length filter:", len(TR_FILTERED))
print("TR retained (maximum 250 monomers):", len(TR_FILTERED_250))
print("TR excluded:", len(TR_FILTERED) - len(TR_FILTERED_250))
print("Maximum monomer count retained:",
      int(TR_FILTERED_250["N_monomers"].max())
      if len(TR_FILTERED_250) else "none")

if TR_FILTERED_250.empty:
    raise ValueError("No TR candidates remain after the 250-monomer filter.")

In [ ]:
APO_SET = set(APO_INFERITE) | set(APO_NOMINATE)
apo_structures = {}

for root in radici_AB:
    df_root = df_AB_normalized_final[
        (df_AB_normalized_final["Root"] == root) &
        (df_AB_normalized_final["Position_type"] == "BIOLOGICAL")
    ].sort_values("Final_position")

    apo_positions_1based = df_root.loc[
        df_root["Node"].isin(APO_SET), "Final_position"
    ].astype(int).tolist()

    non_apo_positions_1based = df_root.loc[
        ~df_root["Node"].isin(APO_SET), "Final_position"
    ].astype(int).tolist()

    if apo_positions_1based and non_apo_positions_1based:
        apo_nodes = dict(zip(
            df_root["Final_position"].astype(int),
            df_root["Node"]
        ))
        apo_structures[root] = (
            [p - 1 for p in apo_positions_1based],
            [p - 1 for p in non_apo_positions_1based],
            apo_nodes
        )

apo_results = []

for tr_index, (_, tr) in enumerate(TR_FILTERED_250.iterrows()):
    period = int(tr["Period"])
    sequence = tr["Complete_sequence"]

    monomer_matrix = np.frombuffer(
        sequence.encode("ascii"), dtype=np.uint8
    ).reshape(-1, period)

    n_monomers = len(monomer_matrix)
    tr_id = f"TR_{tr['Chromosome']}_{tr['Start']}_{tr['End']}"

    for root, (apo_positions, non_apo_positions, apo_nodes) in apo_structures.items():
        maximum_position = max(apo_positions + non_apo_positions)
        n_offsets = n_monomers - maximum_position

        if n_offsets <= 0:
            continue

        apo_array = np.asarray(apo_positions, dtype=int)
        non_apo_array = np.asarray(non_apo_positions, dtype=int)

        for offset in range(n_offsets):
            non_apo_monomers = monomer_matrix[offset + non_apo_array]

            for block_start in range(0, len(apo_array), 8):
                block_positions = apo_array[block_start:block_start + 8]
                apo_monomers = monomer_matrix[offset + block_positions]

                identities = np.mean(
                    apo_monomers[:, None, :] ==
                    non_apo_monomers[None, :, :],
                    axis=2
                )
                recognized = np.all(identities < 0.50, axis=1)

                for i, is_recognized in enumerate(recognized):
                    if is_recognized:
                        zero_based_position = int(block_positions[i])
                        one_based_position = zero_based_position + 1
                        apo_results.append({
                            "TR_ID": tr_id,
                            "TR_Index": tr_index,
                            "Chromosome": tr["Chromosome"],
                            "Start": tr["Start"],
                            "End": tr["End"],
                            "Period": period,
                            "N_monomers": n_monomers,
                            "Root": root,
                            "Offset": offset,
                            "APO": apo_nodes[one_based_position],
                            "APO_position": one_based_position
                        })

    if (tr_index + 1) % 100 == 0 or tr_index == len(TR_FILTERED_250) - 1:
        print(
            f"TR processed: {tr_index + 1}/{len(TR_FILTERED_250)}"
            f" | individual APO recognitions: {len(apo_results)}",
            flush=True
        )

APO_RESULT_COLUMNS = [
    "TR_ID", "TR_Index", "Chromosome", "Start", "End",
    "Period", "N_monomers", "Root", "Offset", "APO", "APO_position"
]
df_APO_individual_250 = pd.DataFrame(apo_results, columns=APO_RESULT_COLUMNS)
APO_OUTPUT = RESULTS_DIR / 'AB_APO_RECOGNITIONS_MAX250.csv'
df_APO_individual_250.to_csv(APO_OUTPUT, index=False)

print("\nAPO scan completed.")
print("Individual recognitions:", len(df_APO_individual_250))
print("Distinct APO nodes recognized:",
      df_APO_individual_250["APO"].nunique()
      if not df_APO_individual_250.empty else 0)
print("Saved:", APO_OUTPUT)

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

apo_universe_df = (
    df_AB_normalized_final[
        df_AB_normalized_final["State"] == "APO"
    ][["Root", "Node"]]
    .drop_duplicates()
    .rename(columns={"Node": "APO"})
)
apo_universe = set(zip(
    apo_universe_df["Root"], apo_universe_df["APO"]
))

apo_associations = (
    df_APO_individual_250[
        ["TR_ID", "Root", "APO", "APO_position"]
    ]
    .drop_duplicates()
    .merge(apo_universe_df, on=["Root", "APO"], how="inner")
)

if apo_associations.empty:
    coverage_by_tr = {}
else:
    coverage_by_tr = (
        apo_associations.groupby("TR_ID")
        .apply(lambda group: set(zip(group["Root"], group["APO"])))
        .to_dict()
    )
    coverage_by_tr = {tr_id: targets for tr_id, targets in coverage_by_tr.items() if targets}

covered_by_any_tr = (
    set().union(*coverage_by_tr.values()) if coverage_by_tr else set()
)
uncovered_apo = apo_universe - covered_by_any_tr

print("APO targets:", len(apo_universe))
print("TR with valid APO associations:", len(coverage_by_tr))
print("Valid TR–APO associations:", len(apo_associations))
print("APO targets without an association:", len(uncovered_apo))

if uncovered_apo:
    print("Uncovered APO targets:", len(uncovered_apo))
    pd.DataFrame(sorted(uncovered_apo), columns=["Root", "APO"]).to_csv(
        RESULTS_DIR / "AB_APO_UNCOVERED_TARGETS.csv", index=False
    )
elif coverage_by_tr:
    tr_ids = sorted(coverage_by_tr)
    apo_list = sorted(apo_universe)
    coverage_matrix = lil_matrix(
        (len(apo_list), len(tr_ids)), dtype=float
    )

    for j, tr_id in enumerate(tr_ids):
        for i, apo in enumerate(apo_list):
            if apo in coverage_by_tr[tr_id]:
                coverage_matrix[i, j] = 1

    solution = milp(
        c=np.ones(len(tr_ids)),
        integrality=np.ones(len(tr_ids)),
        bounds=Bounds(
            np.zeros(len(tr_ids)),
            np.ones(len(tr_ids))
        ),
        constraints=LinearConstraint(
            coverage_matrix.tocsr(),
            np.ones(len(apo_list)),
            np.full(len(apo_list), np.inf)
        ),
        options={"time_limit": 120}
    )

    if solution.success:
        selected_tr = [
            tr_ids[i] for i, value in enumerate(solution.x)
            if value > 0.5
        ]
        print("\nMinimum TR set for complete APO coverage:", len(selected_tr))
        display(
            df_APO_individual_250[
                df_APO_individual_250["TR_ID"].isin(selected_tr)
            ][
                ["TR_ID", "Chromosome", "Start", "End",
                 "Period", "N_monomers"]
            ].drop_duplicates("TR_ID").reset_index(drop=True)
        )
    else:
        print("The minimum-set optimization did not complete:", solution.message)
else:
    print("No APO associations were found.")

In [ ]:
stop_apo_positions = {}
divide_positions = {}

for root in radici_AB:
    df_root = (
        df_AB_normalized_final[
            df_AB_normalized_final["Root"] == root
        ]
        .sort_values("Final_position")
        .reset_index(drop=True)
    )

    biological_indices = [
        i for i, row in df_root.iterrows()
        if row["Position_type"] == "BIOLOGICAL"
    ]

    stop_apo_positions[root] = [
        i for i in biological_indices
        if df_root.loc[i, "State"] in {"STOP", "APO"}
    ]
    divide_positions[root] = [
        i for i in biological_indices
        if df_root.loc[i, "State"] == "DIVIDE"
    ]

n_stop_apo = sum(map(len, stop_apo_positions.values()))
n_divide = sum(map(len, divide_positions.values()))

assert len(radici_AB) == 32
assert n_stop_apo == 487, n_stop_apo
assert n_divide == 455, n_divide

print("AB roots:", len(radici_AB))
print("STOP + APO positions:", n_stop_apo)
print("DIVIDE positions:", n_divide)
print("TR candidates:", len(TR_FILTERED_250))

stop_divide_results = []
identity_threshold = 0.50

for tr_index, (_, tr) in enumerate(TR_FILTERED_250.iterrows()):
    period = int(tr["Period"])
    sequence = tr["Complete_sequence"]

    monomer_matrix = np.frombuffer(
        sequence.encode("ascii"), dtype=np.uint8
    ).reshape(-1, period)

    n_monomers = len(monomer_matrix)
    tr_id = f"TR_{tr['Chromosome']}_{tr['Start']}_{tr['End']}"

    for root in radici_AB:
        left_positions = stop_apo_positions[root]
        right_positions = divide_positions[root]

        if not left_positions or not right_positions:
            continue

        maximum_position = max(left_positions + right_positions)
        n_offsets = n_monomers - maximum_position
        if n_offsets <= 0:
            continue

        right_array = np.asarray(right_positions, dtype=int)

        for offset in range(n_offsets):
            divide_monomers = monomer_matrix[offset + right_array]
            window_passes = True

            for position in left_positions:
                stop_apo_monomer = monomer_matrix[offset + position]
                identities = np.count_nonzero(
                    divide_monomers == stop_apo_monomer,
                    axis=1
                ) / period

                if np.any(identities >= identity_threshold):
                    window_passes = False
                    break

            if window_passes:
                stop_divide_results.append({
                    "TR_ID": tr_id,
                    "TR_Index": tr_index,
                    "Chromosome": tr["Chromosome"],
                    "Start": tr["Start"],
                    "End": tr["End"],
                    "Period": period,
                    "N_monomers": n_monomers,
                    "Root": root,
                    "Offset": offset
                })

    if (tr_index + 1) % 100 == 0 or tr_index == len(TR_FILTERED_250) - 1:
        print(
            f"TR processed: {tr_index + 1}/{len(TR_FILTERED_250)}"
            f" | STOP+APO/DIVIDE recognitions: {len(stop_divide_results)}",
            flush=True
        )

RESULT_COLUMNS = [
    "TR_ID", "TR_Index", "Chromosome", "Start", "End",
    "Period", "N_monomers", "Root", "Offset"
]
df_STOP_APO_vs_DIVIDE_250 = pd.DataFrame(
    stop_divide_results, columns=RESULT_COLUMNS
)

STOP_DIVIDE_OUTPUT = RESULTS_DIR / 'AB_STOP_APO_vs_DIVIDE_MAX250.csv'
df_STOP_APO_vs_DIVIDE_250.to_csv(
    STOP_DIVIDE_OUTPUT, index=False
)

print("\nSTOP + APO versus DIVIDE scan completed.")
print("Recognized windows:", len(df_STOP_APO_vs_DIVIDE_250))
print("Distinct TR:", df_STOP_APO_vs_DIVIDE_250["TR_ID"].nunique())
print("Roots represented:", df_STOP_APO_vs_DIVIDE_250["Root"].nunique())
print("Saved:", STOP_DIVIDE_OUTPUT)

## 4. AB results

The AB scan writes two independent tables into the common results directory:

- `AB_APO_RECOGNITIONS_MAX250.csv`
- `AB_STOP_APO_vs_DIVIDE_MAX250.csv`

The minimum-TR APO coverage optimization is reported separately from the exhaustive recognition table.

In [ ]:
print("Result files:")
for output_file in [APO_OUTPUT, STOP_DIVIDE_OUTPUT]:
    if output_file.exists():
        print(f"- {output_file.name}: {output_file.stat().st_size:,} bytes")
    else:
        print(f"- {output_file.name}: not created")

print('AB result tables are stored in:', RESULTS_DIR)

## Interpretation notes

- `TR_CATALOG_RAW.tsv` is the authoritative input catalog.
- Incomplete terminal monomers are discarded before scanning.
- TRs with more than 250 complete monomers are excluded from both analyses.
- `NULL` is a virtual position, not a biological STOP or DIVIDE state; it is excluded from biological comparisons.
- The APO-only scan and the STOP+APO-versus-DIVIDE scan are distinct analyses and their result tables should not be merged.
- CSV output positions for APO are reported as one-based normalized positions; offsets are zero-based.

## 5. MS, C, D and E — lineage reconstruction and normalization

The lineage-specific normalization rules are retained from the cleaned MS/C/D/E workflow. MS uses its 16 established four-letter subtrees; C uses its eight three-letter roots; D and E are each analyzed as one complete tree. Internal virtual positions remain `NULL` and are excluded from biological comparisons.

In [ ]:
# 2. Imports and validated TR catalogue
import re
import math
import json
import numpy as np
import pandas as pd
from collections import deque

TR_RAW = pd.read_csv(TR_FILE, sep='\t', low_memory=False)
TR_RAW['RAW_INDEX'] = TR_RAW.index
required_columns = {'Chromosome','Start','End','Period','Repeat_Sequence'}
missing_columns = required_columns - set(TR_RAW.columns)
if missing_columns:
    raise ValueError(f'TR_CATALOG_RAW.tsv is missing columns: {sorted(missing_columns)}')
TR_RAW['Period'] = pd.to_numeric(TR_RAW['Period'], errors='coerce')
TR_RAW['Repeat_Sequence'] = TR_RAW['Repeat_Sequence'].fillna('').astype(str).str.upper()
TR_RAW = TR_RAW[TR_RAW['Period'].notna() & (TR_RAW['Period'] > 0)].copy()
TR_RAW['Period'] = TR_RAW['Period'].astype(int)
TR_RAW['Repeat_Length'] = TR_RAW['Repeat_Sequence'].str.len()
TR_RAW['N_monomeri'] = TR_RAW['Repeat_Length'] // TR_RAW['Period']
TR_RAW['Coda_esclusa'] = TR_RAW['Repeat_Length'] % TR_RAW['Period']
TR_RAW['Sequenza_completa'] = [
    seq[:(len(seq)//periodo)*periodo]
    for seq, periodo in zip(TR_RAW['Repeat_Sequence'], TR_RAW['Period'])
]
print(f'Validated raw TR rows: {len(TR_RAW):,}')
print('TRs with at least one complete monomer:', int((TR_RAW['N_monomeri'] >= 1).sum()))


In [ ]:
# 3. Lineage parser
SECTION_MARKERS = {
    'MS': ['========== STOP (80) ==========', '========== DIVIDE (93) ==========', '========== APO (14) ==========' ],
    'C':  ['========== STOP (47) ==========', '========== DIVIDE (47) ==========', '========== APO (1) ==========' ],
    'D':  ['========== STOP (20) ==========', '========== DIVIDE (19) ==========', '========== APO (0) ==========' ],
    'E':  ['========== STOP (20) ==========', '========== DIVIDE (19) ==========', '========== APO (0) ==========' ],
}
FILES = {k: INPUT_FILES[f'{k}_BOTTOM_UP_TOP_DOWN.txt'] for k in SECTION_MARKERS}

def _section(lines, title, all_markers):
    if title not in lines:
        raise ValueError(f'Section marker not found: {title}')
    start = lines.index(title) + 1
    later = [lines.index(m) for m in all_markers if m in lines and lines.index(m) >= start]
    end = min(later) if later else len(lines)
    return [x.strip() for x in lines[start:end] if x.strip()]

def read_lineage(name):
    path = FILES[name]
    lines = [x.strip() for x in path.read_text(encoding='utf-8-sig').splitlines()]
    markers = SECTION_MARKERS[name]
    tree_marker = '-- ALBERO COMPLETO --'
    if tree_marker not in lines:
        raise ValueError(f'{name}: complete-tree marker not found')
    tree_end = lines.index(markers[0])
    tree_lines = lines[lines.index(tree_marker)+1:tree_end]
    tree = set()
    for line in tree_lines:
        if line == name:
            tree.add('')
        elif line.startswith(name):
            tree.add(line[len(name):].strip())
    data = {}
    labels = ['STOP','DIVIDE','APO']
    for label, marker in zip(labels, markers):
        rows = _section(lines, marker, markers)
        names = []
        for row in rows:
            if row == name:
                names.append('')
            elif row.startswith(name):
                names.append(row[len(name):].strip())
        data[label] = set(names)
    if not tree:
        raise ValueError(f'{name}: no biological nodes parsed')
    assigned = data['STOP'] | data['DIVIDE'] | data['APO']
    duplicates = sum(len(data[k]) for k in data) - len(assigned)
    if duplicates:
        raise ValueError(f'{name}: a node occurs in more than one state section')
    unclassified = tree - assigned
    outside = assigned - tree
    if unclassified or outside:
        raise ValueError(f'{name}: unclassified tree nodes={len(unclassified)}; state nodes outside tree={len(outside)}')
    return {'tree': tree, **data}

LINEAGES = {name: read_lineage(name) for name in FILES}
EXPECTED = {'MS': (80,93,14), 'C': (47,47,1), 'D': (20,19,0), 'E': (20,19,0)}
for name, data in LINEAGES.items():
    observed = tuple(len(data[k]) for k in ('STOP','DIVIDE','APO'))
    if observed != EXPECTED[name]:
        raise ValueError(f'{name}: expected STOP/DIVIDE/APO={EXPECTED[name]}, observed={observed}')
    print(f'{name}: tree={len(data["tree"])} | STOP={observed[0]} | DIVIDE={observed[1]} | APO={observed[2]}')


In [ ]:
# 4. Normalization helpers
MS_ROOTS = ['aaaa','aaap','aapa','aapp','apaa','apap','appa','appp',
            'paaa','paap','papa','papp','ppaa','ppap','pppa','pppp']
C_ROOTS = ['aaa','aap','apa','app','paa','pap','ppa','ppp']
C_CHILD_ORDER = ('a','p','l','r','d','v')

def state_of(data, node):
    for state in ('DIVIDE','STOP','APO'):
        if node in data[state]:
            return state
    raise ValueError(f'Node has no explicit state: {node!r}')

def normalize_ms(data):
    rows=[]
    for root in MS_ROOTS:
        nodes=sorted((n for n in data['tree'] if n.startswith(root)), key=lambda n:(len(n),n))
        if not nodes: raise ValueError(f'MS subtree missing: {root}')
        max_depth=max(len(n) for n in nodes)
        positions=[]
        for node in nodes:
            state=state_of(data,node)
            positions.append({'Radice':root,'Nodo':node,'Stato':state,'Codice': '0' if state=='DIVIDE' else '1' if state=='STOP' else 'APO','Tipo_posizione':'BIOLOGICA'})
            if state=='APO' or (state=='STOP' and len(node)<max_depth):
                for j in (1,2): positions.append({'Radice':root,'Nodo':f'{node}_NULL{j}','Stato':'NULL','Codice':'NULL','Tipo_posizione':'VIRTUALE'})
        for i,row in enumerate(positions,1): row['Posizione_finale']=i; rows.append(row)
    return pd.DataFrame(rows)

def normalize_binary_tree(name, data, roots, child_order):
    rows=[]
    for root in roots:
        nodes={n for n in data['tree'] if n.startswith(root)}
        if root not in nodes: raise ValueError(f'{name}: expected subtree root not found: {root}')
        max_depth=max(len(n)-len(root) for n in nodes)
        queue=deque([(root,0,0)])
        positions=[]
        while queue:
            node, depth, index=queue.popleft()
            if node is None:
                positions.append({'Radice':root,'Nodo':'NULL','Stato':'NULL','Codice':'NULL','Tipo_posizione':'VIRTUALE','Profondita':depth,'Posizione':index})
                if depth<max_depth:
                    queue.extend([(None,depth+1,2*index+1),(None,depth+1,2*index+2)])
                continue
            state=state_of(data,node)
            positions.append({'Radice':root,'Nodo':node,'Stato':state,'Codice':'0' if state=='DIVIDE' else '1' if state=='STOP' else 'APO','Tipo_posizione':'BIOLOGICA','Profondita':depth,'Posizione':index})
            if depth<max_depth:
                if state=='DIVIDE':
                    children=[node+s for s in child_order if node+s in nodes]
                    if len(children)!=2: raise ValueError(f'{name}: {node!r} has {len(children)} children: {children}')
                    queue.extend([(children[0],depth+1,2*index+1),(children[1],depth+1,2*index+2)])
                else:
                    queue.extend([(None,depth+1,2*index+1),(None,depth+1,2*index+2)])
        positions.sort(key=lambda x:x['Posizione'])
        while positions and positions[-1]['Stato']=='NULL': positions.pop()
        for i,row in enumerate(positions,1): row['Posizione_finale']=i; rows.append(row)
    return pd.DataFrame(rows)

NORMALIZED = {
    'MS': normalize_ms(LINEAGES['MS']),
    'C': normalize_binary_tree('C', LINEAGES['C'], C_ROOTS, C_CHILD_ORDER),
    'D': normalize_binary_tree('D', LINEAGES['D'], [''], tuple('abcdefghijklmnopqrstuvwxyz')) ,
    'E': normalize_binary_tree('E', LINEAGES['E'], [''], tuple('abcdefghijklmnopqrstuvwxyz')),
}


In [ ]:
# 5. Summaries and consistency checks
summary=[]
for name, df in NORMALIZED.items():
    bio=df[df['Tipo_posizione']=='BIOLOGICA']
    summary.append({'Lineage':name,'Subtrees':bio['Radice'].nunique(),'Biological_nodes':len(bio),
                    'DIVIDE':int((bio.Stato=='DIVIDE').sum()),'STOP':int((bio.Stato=='STOP').sum()),
                    'APO':int((bio.Stato=='APO').sum()),'Internal_NULL':int((df.Stato=='NULL').sum()),
                    'Normalized_positions':len(df)})
SUMMARY=pd.DataFrame(summary)
EXPECTED_APO={'MS':14,'C':1,'D':0,'E':0}
for name, expected in EXPECTED_APO.items():
    observed=int((NORMALIZED[name]['Stato']=='APO').sum())
    if observed!=expected: raise ValueError(f'{name}: expected {expected} APO, found {observed}')
print('Validated APO counts:', EXPECTED_APO)
display(SUMMARY)


In [ ]:
# 6. Prepare complete monomer matrices from TR_CATALOG_RAW
def tr_matrix(row):
    period=int(row['Period'])
    seq=str(row['Sequenza_completa'])
    if len(seq)%period: raise ValueError('Incomplete monomer sequence after trimming')
    return np.frombuffer(seq.encode('ascii'), dtype=np.uint8).reshape(-1,period)

MAX_MONOMERS = 250

# Apply the shared project TR criteria to the raw catalog.
TR_BASE = TR_RAW[
    (TR_RAW['Period'] > 1) &
    (TR_RAW['N_monomeri'] > 0) &
    (TR_RAW['N_monomeri'] <= MAX_MONOMERS)
].copy()

short_period = TR_BASE[TR_BASE['Period'].isin([2, 3])].copy()
short_period['A_in_every_monomer'] = [
    all('A' in seq[i:i + period]
        for i in range(0, len(seq), period))
    for seq, period in zip(short_period['Sequenza_completa'], short_period['Period'])
]
TR_CATALOG = pd.concat([
    TR_BASE[TR_BASE['Period'] >= 4],
    short_period[short_period['A_in_every_monomer']]
], ignore_index=True)
TR_CATALOG['TR_ID'] = [
    f"TR_{r.Chromosome}_{r.Start}_{r.End}"
    for r in TR_CATALOG.itertuples()
]
print('Raw TR rows:', len(TR_RAW))
print('TR candidates (Period > 1, complete monomers, max 250, period-2/3 A rule):', len(TR_CATALOG))
print('Maximum complete monomers retained:', int(TR_CATALOG['N_monomeri'].max()) if len(TR_CATALOG) else 0)


## 7. Independent scans: STOP vs DIVIDE, and APO vs non-APO

A window is promoted only when every cross-class monomer comparison has identity **below** the threshold. `NULL` positions are excluded. The two scans are independent: the STOP/DIVIDE scan does not include APO; the APO/non-APO scan compares APO against all other biological nodes.

In [ ]:
# 7. Scan functions
IDENTITY_THRESHOLD=0.50

def scan_pairwise(lineage, df_norm, class_a, class_b, threshold=IDENTITY_THRESHOLD):
    results=[]
    for root, group in df_norm.groupby('Radice', sort=True):
        ordered=group.sort_values('Posizione_finale').reset_index(drop=True)
        # Keep virtual NULL slots in the coordinate system; exclude them only from class membership.
        pos_a=[i for i,row in ordered.iterrows() if row['Tipo_posizione']=='BIOLOGICA' and row['Stato']==class_a]
        pos_b=[i for i,row in ordered.iterrows() if row['Tipo_posizione']=='BIOLOGICA' and row['Stato']==class_b]
        if not pos_a or not pos_b: continue
        required=max(pos_a+pos_b)+1
        # Restrict each root to TRs long enough for this normalized window.
        eligible_tr = TR_CATALOG[TR_CATALOG['N_monomeri'] >= required]
        pos_b_array = np.asarray(pos_b, dtype=int)
        period_cache = None
        for _,tr in eligible_tr.iterrows():
            n=int(tr['N_monomeri'])
            period=int(tr['Period'])
            matrix=tr_matrix(tr)
            for offset in range(n-required+1):
                window=matrix[offset:offset+required]
                passed=True
                for ia in pos_a:
                    identities = np.count_nonzero(
                        window[pos_b_array] == window[ia], axis=1
                    ) / period
                    if np.any(identities >= threshold):
                        passed=False
                        break
                if passed:
                    results.append({'Lineage':lineage,'Radice':root,'Comparison':f'{class_a}_vs_{class_b}',
                                    'TR_ID':tr['TR_ID'],'TR_Indice_RAW':int(tr['RAW_INDEX']),'Chromosome':tr['Chromosome'],
                                    'Start':tr['Start'],'End':tr['End'],'Period':period,
                                    'N_monomeri':n,'Window_length':required,'Offset':offset})
    columns = ['Lineage','Radice','Comparison','TR_ID','TR_Indice_RAW','Chromosome',
               'Start','End','Period','N_monomeri','Window_length','Offset']
    return pd.DataFrame(results, columns=columns)

RESULTS_STOP_DIVIDE=[]
RESULTS_APO=[]
for lineage, df in NORMALIZED.items():
    print(f'Running {lineage}...')
    RESULTS_STOP_DIVIDE.append(scan_pairwise(lineage,df,'STOP','DIVIDE'))
    # APO is compared with every other biological position; if APO is absent, return an empty result.
    if (df['Stato']=='APO').any():
        temp=df.copy()
        temp['Stato_APO_SCAN']=np.where(temp['Stato']=='APO','APO',np.where(temp['Tipo_posizione']=='BIOLOGICA','NON_APO','NULL'))
        # Scan helper uses the original biological state column; make a scan-specific copy.
        temp.loc[temp['Tipo_posizione']=='BIOLOGICA','Stato']=temp.loc[temp['Tipo_posizione']=='BIOLOGICA','Stato_APO_SCAN']
        RESULTS_APO.append(scan_pairwise(lineage,temp,'APO','NON_APO'))
    else:
        RESULTS_APO.append(pd.DataFrame())

DF_STOP_DIVIDE=pd.concat(RESULTS_STOP_DIVIDE,ignore_index=True)
DF_APO=pd.concat(RESULTS_APO,ignore_index=True)
print('STOP vs DIVIDE windows:',len(DF_STOP_DIVIDE))
print('APO vs non-APO windows:',len(DF_APO))


In [ ]:
# 8. Save outputs
OUT=RESULTS_DIR/'MS_C_D_E'
OUT.mkdir(exist_ok=True)
SUMMARY.to_csv(OUT/'LINEAGE_SUMMARY.csv',index=False)
for lineage,df in NORMALIZED.items():
    df.to_csv(OUT/f'{lineage}_NORMALIZED_POSITIONS.csv',index=False)
DF_STOP_DIVIDE.to_csv(OUT/'TR_STOP_vs_DIVIDE.csv',index=False)
DF_APO.to_csv(OUT/'TR_APO_vs_NON_APO.csv',index=False)
print('Saved outputs to:',OUT)


## 6. Unified summary and download

The final cell collects the five lineage summaries, checks the expected APO counts, creates a ZIP containing every CSV result, and downloads that ZIP.

In [ ]:
# Build a single lineage summary table and package all result CSVs.
import zipfile

all_summary = []
all_summary.append({
    'Lineage': 'AB',
    'Full_tree_nodes': len(albero_completo),
    'Full_tree_DIVIDE': len(DIVIDE_AB),
    'Full_tree_STOP': len(STOP_AB),
    'Full_tree_APO': len(APO_INFERITE) + len(APO_NOMINATE),
    'Normalized_subtrees': len(radici_AB),
    'Normalized_biological_nodes': biological_count,
    'Normalized_DIVIDE': divide_count,
    'Normalized_STOP': stop_count,
    'Normalized_APO': apo_count,
    'Internal_NULL': int((df_AB_normalized_final['Position_type'] == 'VIRTUAL').sum())
})
for name, data in LINEAGES.items():
    df = NORMALIZED[name]
    bio = df[df['Tipo_posizione'] == 'BIOLOGICA']
    all_summary.append({
        'Lineage': name,
        'Full_tree_nodes': len(data['tree']),
        'Full_tree_DIVIDE': len(data['DIVIDE']),
        'Full_tree_STOP': len(data['STOP']),
        'Full_tree_APO': len(data['APO']),
        'Normalized_subtrees': int(bio['Radice'].nunique()),
        'Normalized_biological_nodes': len(bio),
        'Normalized_DIVIDE': int((bio['Stato'] == 'DIVIDE').sum()),
        'Normalized_STOP': int((bio['Stato'] == 'STOP').sum()),
        'Normalized_APO': int((bio['Stato'] == 'APO').sum()),
        'Internal_NULL': int((df['Stato'] == 'NULL').sum())
    })

UNIFIED_SUMMARY = pd.DataFrame(all_summary)
expected_apo = {'AB': 96, 'MS': 14, 'C': 1, 'D': 0, 'E': 0}
observed_apo = dict(zip(UNIFIED_SUMMARY['Lineage'], UNIFIED_SUMMARY['Full_tree_APO']))
if observed_apo != expected_apo:
    raise ValueError(f'APO summary mismatch: {observed_apo}')

UNIFIED_SUMMARY.to_csv(RESULTS_DIR / 'LINEAGE_SUMMARY_AB_MS_C_D_E.csv', index=False)
df_AB_32_verification.to_csv(RESULTS_DIR / 'AB_32_ROOT_VERIFICATION.csv', index=False)
df_AB_normalized_final.to_csv(RESULTS_DIR / 'AB_NORMALIZED_POSITIONS.csv', index=False)
display(UNIFIED_SUMMARY)

zip_path = BASE / 'STOP_APO_AB_MS_C_D_E_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.rglob('*.csv')):
        archive.write(path, arcname=str(path.relative_to(RESULTS_DIR)))

print('Result CSV files:', len(list(RESULTS_DIR.rglob('*.csv'))))
print('ZIP archive:', zip_path, f'({zip_path.stat().st_size:,} bytes)')
files.download(str(zip_path))
